# CoT-Driven RAG Demo
- **Retrieval**: ranking, context windows, and relevance scoring
- **CoT (Chain-of-Thought)**: Structured reasoning with evidence citation
- **Knowledge Base**: Example pathway knowledge files (generated by Claude 4.5) for demo

In [22]:
import os
import sys
import pandas as pd
import json

# Setup path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from mozzarellm import (
    analyze_gene_clusters,
    reshape_to_clusters,
    ENHANCED_COT_INSTRUCTIONS,
    CONCISE_COT_INSTRUCTIONS,
)
from mozzarellm.prompts import ROBUST_SCREEN_CONTEXT, ROBUST_CLUSTER_PROMPT
# # Load secrets from .env if present (expects ANTHROPIC_API_KEY when using Claude)
from dotenv import load_dotenv
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# # Load secrets from .env if present (expects ANTHROPIC_API_KEY when using Claude)
load_dotenv()
print('ANTHROPIC set?', 'ANTHROPIC_API_KEY' in os.environ)
from mozzarellm.configs import DEFAULT_ANTHROPIC_CONFIG

# Set your API key (or use .env file)
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"

ANTHROPIC set? True


## Load Sample Data

In [23]:
# Load sample data
sample_csv = os.path.join(project_root, 'examples', 'sample_data.csv')
sample_data = pd.read_csv(sample_csv)

cluster_df, gene_features = reshape_to_clusters(
    input_df=sample_data,
    uniprot_col='uniprot_function',
    verbose=True
)

print(f"\nClusters: {len(cluster_df)}")
print(f"Genes with annotations: {len(gene_features)}")
display(cluster_df.head())

Using provided DataFrame with 140 rows
Found 140 genes across 6 clusters
Extracting gene features from uniprot_function column

Clusters: 6
Genes with annotations: 140


,cluster_id,genes
0,21,AATF;ABT1;BYSL;BMS1;C1orf131;EIF3M;EIF4A1;ESF1...
1,37,SRSF3;PDPK1;RICTOR;RPTOR;SEH1L;SGF29;PRKAR1A;P...
2,121,CCDC174;FAM32A;GABPA;SP2;N6AMT1;SETD2;SON;POU5...
3,149,KRAS;BRAF;NDUFV2;NDUFA6;NDUFC1;RAD23B;SNAPC1;N...
4,167,POMP;PSMA2;PSMB7;PSMB3;PSMA7;PSMB2;PSMA1;PSMA4...


## Comparison: Without vs. With RAG

### Test 1: Baseline (No RAG, No CoT)

In [24]:
# Baseline: No retrieval, no structured CoT
results_baseline = analyze_gene_clusters(
    input_df=cluster_df.head(2),  # Just 2 clusters for demo
    model_name='claude-3-7-sonnet-20250219',
    config_dict=DEFAULT_ANTHROPIC_CONFIG,
    screen_context=ROBUST_SCREEN_CONTEXT,
    cluster_analysis_prompt=ROBUST_CLUSTER_PROMPT,
    gene_annotations_df=gene_features,
    batch_size=1,
    save_outputs=False,
    # RAG and CoT disabled
    use_retrieval=False,
    cot_instructions=None,
)

print("\n=== BASELINE RESULTS ===")
display(results_baseline['cluster_df'])

INFO:cluster_analysis_20251019_215752.log:Processing 2 clusters with model claude-3-7-sonnet-20250219


Loaded data with 2 rows and columns: ['cluster_id', 'genes']
Created annotations dictionary with 140 entries from DataFrame


Processing clusters:   0%|          | 0/2 [00:00<?, ?it/s]INFO:cluster_analysis_20251019_215752.log:Using Anthropic Claude API


Using provided template string
Appending output format instructions to template
Added 39 gene feature descriptions to prompt


INFO:cluster_analysis_20251019_215752.log:Anthropic API call successful: model=claude-3-7-sonnet-20250219
INFO:cluster_analysis_20251019_215752.log:Success for cluster 21
Processing clusters:  50%|█████     | 1/2 [00:19<00:19, 19.69s/it]INFO:cluster_analysis_20251019_215752.log:Using Anthropic Claude API


Using provided template string
Appending output format instructions to template
Added 33 gene feature descriptions to prompt


INFO:cluster_analysis_20251019_215752.log:Anthropic API call successful: model=claude-3-7-sonnet-20250219
INFO:cluster_analysis_20251019_215752.log:Success for cluster 37
Processing clusters: 100%|██████████| 2/2 [00:41<00:00, 20.95s/it]
INFO:cluster_analysis_20251019_215752.log:Completed analysis for 2 clusters without saving to disk



=== BASELINE RESULTS ===


,cluster_id,cluster_biological_process,pathway_confidence_level,cluster_importance_score,follow_up_suggestion,established_genes,established_gene_count,uncharacterized_genes,uncharacterized_gene_count,novel_role_genes,...,total_gene_count,highest_unchar_importance,average_unchar_importance,high_unchar_genes,high_unchar_gene_count,highest_novel_role_importance,average_novel_role_importance,high_novel_role_genes,high_novel_role_gene_count,all_cluster_genes
0,21,Ribosome biogenesis and pre-rRNA processing,High,2.64,Cluster 21 shows a strong signature of ribosom...,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;NOP1...,17,NOC4L;NOL10;RRP12,3,DYRK1A;INO80;NEPRO;ZCCHC9,...,24,7,6.67,,0,8,6.75,DYRK1A:8,1,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;NOP1...
1,37,mTOR signaling pathway,High,2.64,This cluster shows a strong enrichment for com...,MTOR;RPTOR;RICTOR;MLST8;RHEB;PDPK1;WDR24;SEH1L,8,C7orf26,1,SLC4A7;TRAPPC8;TRAPPC11;TRAPPC3;COG2;COG3;COG4,...,16,5,5.00,,0,8,5.86,SLC4A7:8,1,MTOR;RPTOR;RICTOR;MLST8;RHEB;PDPK1;WDR24;SEH1L...


### Test 2: With Enhanced RAG + Structured CoT

In [25]:
# Enhanced: With retrieval and structured CoT
knowledge_dir = os.path.join(project_root, 'data', 'knowledge')
print(f"Knowledge directory: {knowledge_dir}")
print(f"Knowledge files available: {os.listdir(knowledge_dir)}")

results_enhanced = analyze_gene_clusters(
    input_df=cluster_df.head(2),  # Same 2 clusters
    model_name='claude-3-7-sonnet-20250219',
    config_dict=DEFAULT_ANTHROPIC_CONFIG,
    screen_context=ROBUST_SCREEN_CONTEXT,
    cluster_analysis_prompt=ROBUST_CLUSTER_PROMPT,
    gene_annotations_df=gene_features,
    batch_size=1,
    save_outputs=False,
    # RAG and CoT enabled
    use_retrieval=True,
    knowledge_dir=knowledge_dir,
    retriever_k=15,  # Retrieve more snippets
    cot_instructions=ENHANCED_COT_INSTRUCTIONS,  # Structured reasoning
)

print("\n=== ENHANCED RAG + CoT RESULTS ===")
display(results_enhanced['cluster_df'])

INFO:cluster_analysis_20251019_215834.log:Processing 2 clusters with model claude-3-7-sonnet-20250219


Knowledge directory: c:\Users\alexa\WORKSPACE\UROP\mozzarellm\data\knowledge
Knowledge files available: ['mtor_signaling.md', 'proteasome_pathway.md', 'readme.md', 'ribosome_biogenesis.md']
Loaded data with 2 rows and columns: ['cluster_id', 'genes']
Created annotations dictionary with 140 entries from DataFrame


Processing clusters:   0%|          | 0/2 [00:00<?, ?it/s]INFO:cluster_analysis_20251019_215834.log:Using Anthropic Claude API


Using provided template string
Appending output format instructions to template
Added 39 gene feature descriptions to prompt


INFO:cluster_analysis_20251019_215834.log:Anthropic API call successful: model=claude-3-7-sonnet-20250219
INFO:cluster_analysis_20251019_215834.log:Success for cluster 21
Processing clusters:  50%|█████     | 1/2 [00:32<00:32, 32.23s/it]INFO:cluster_analysis_20251019_215834.log:Using Anthropic Claude API


Using provided template string
Appending output format instructions to template
Added 33 gene feature descriptions to prompt


INFO:cluster_analysis_20251019_215834.log:Anthropic API call successful: model=claude-3-7-sonnet-20250219
INFO:cluster_analysis_20251019_215834.log:Success for cluster 37
Processing clusters: 100%|██████████| 2/2 [00:50<00:00, 25.43s/it]
INFO:cluster_analysis_20251019_215834.log:Completed analysis for 2 clusters without saving to disk



=== ENHANCED RAG + CoT RESULTS ===


,cluster_id,cluster_biological_process,pathway_confidence_level,cluster_importance_score,follow_up_suggestion,established_genes,established_gene_count,uncharacterized_genes,uncharacterized_gene_count,novel_role_genes,...,total_gene_count,highest_unchar_importance,average_unchar_importance,high_unchar_genes,high_unchar_gene_count,highest_novel_role_importance,average_novel_role_importance,high_novel_role_genes,high_novel_role_gene_count,all_cluster_genes
0,21,Ribosome biogenesis and rRNA processing,High,2.10,Cluster 21 shows a strong signature of ribosom...,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;DDX1...,19,NOC4L;NOL10;RRP12,3,EIF3M;EIF4A1;EIF3E;EIF3F;EIF3D;DYRK1A;ABT1;ESF...,...,39,7,6.67,,0,7,4.71,,0,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;DDX1...
1,37,mTOR signaling pathway,High,2.64,Cluster 37 shows a strong enrichment for compo...,MTOR;RPTOR;RICTOR;MLST8;RHEB;PDPK1;SEH1L;WDR24,8,C7orf26,1,SLC4A7;PRKAR1A;TRAPPC8,...,12,5,5.00,,0,8,6.67,SLC4A7:8,1,MTOR;RPTOR;RICTOR;MLST8;RHEB;PDPK1;SEH1L;WDR24...


## Inspect Retrieved Evidence

In [26]:
from mozzarellm.utils.retrieval import retrieve_context

# Test retrieval for first cluster
cluster_genes = cluster_df.iloc[0]['genes'].split(';')
print(f"Cluster genes: {cluster_genes[:5]}... ({len(cluster_genes)} total)")

# Convert gene_features to dict
gene_annotations_dict = dict(zip(gene_features['gene_symbol'], gene_features['uniprot_function']))

# Retrieve context
retrieved = retrieve_context(
    cluster_genes=cluster_genes,
    gene_annotations=gene_annotations_dict,
    screen_context=ROBUST_SCREEN_CONTEXT,
    knowledge_dir=knowledge_dir,
    k=15,
)

print("\n=== RETRIEVAL METADATA ===")
print(json.dumps(retrieved['retrieval_metadata'], indent=2))

print("\n=== TOP RETRIEVED SNIPPETS ===")
for i, snippet in enumerate(retrieved['snippets'][:5], 1):
    print(f"\n[{i}] Source: {snippet['source']} | Relevance: {snippet['relevance_score']}")
    print(f"    {snippet['text'][:200]}...")

Cluster genes: ['AATF', 'ABT1', 'BYSL', 'BMS1', 'C1orf131']... (39 total)

=== RETRIEVAL METADATA ===
{
  "knowledge_dir": "c:\\Users\\alexa\\WORKSPACE\\UROP\\mozzarellm\\data\\knowledge",
  "k": 15,
  "genes_queried": 39,
  "total_retrieved": 41,
  "annotations_found": 36,
  "knowledge_snippets_found": 4
}

=== TOP RETRIEVED SNIPPETS ===

[1] Source: annotations | Relevance: 100
    AATF: Part of the small subunit (SSU) processome, first precursor of the small eukaryotic ribosomal subunit. During the assembly of the SSU processome in the nucleolus, many ribosome biogenesis factor...

[2] Source: annotations | Relevance: 100
    ABT1: Could be a novel TATA-binding protein (TBP) which can function as a basal transcription activator. Can act as a regulator of basal transcription for class II genes (By similarity). {ECO:0000250}...

[3] Source: annotations | Relevance: 100
    BYSL: Required for processing of 20S pre-rRNA precursor and biogenesis of 40S ribosomal subunits. May be required

## Compare Gene Prioritization

Compare how genes are prioritized with and without RAG+CoT:

In [30]:
# Compare gene-level results
baseline_genes = results_baseline['gene_df']
enhanced_genes = results_enhanced['gene_df']

print("=== BASELINE: Top Priority Genes ===")
display(baseline_genes.nlargest(10, 'gene_importance_score')[[
    'gene_name', 'gene_category', 'gene_importance_score', 'gene_description'
]])

print("\n=== ENHANCED RAG+CoT: Top Priority Genes ===")
display(enhanced_genes.nlargest(10, 'gene_importance_score')[[
    'gene_name', 'gene_category', 'gene_importance_score', 'gene_description'
]])

=== BASELINE: Top Priority Genes ===


,gene_name,gene_category,gene_importance_score,gene_description
4,DYRK1A,novel_role,8,DYRK1A is a well-characterized kinase with rol...
8,SLC4A7,novel_role,8,SLC4A7 is a sodium-bicarbonate cotransporter r...
5,INO80,novel_role,7,INO80 is established as a chromatin remodeling...
9,TRAPPC8,novel_role,7,TRAPPC8 is known for its role in ER-to-Golgi t...
0,NOC4L,uncharacterized,7,NOC4L (Nucleolar Complex Associated 4 Homolog)...
1,NOL10,uncharacterized,7,NOL10 (Nucleolar Protein 10) has minimal funct...
6,NEPRO,novel_role,6,NEPRO is characterized for its role in Notch s...
7,ZCCHC9,novel_role,6,ZCCHC9 is known to regulate NF-kB-mediated tra...
10,TRAPPC11,novel_role,6,"TRAPPC11 functions in ER-to-Golgi trafficking,..."
2,RRP12,uncharacterized,6,RRP12 lacks detailed functional annotation in ...



=== ENHANCED RAG+CoT: Top Priority Genes ===


,gene_name,gene_category,gene_importance_score,gene_description
21,SLC4A7,novel_role,8,SLC4A7 is a sodium-bicarbonate cotransporter p...
9,DYRK1A,novel_role,7,Well-characterized dual-specificity kinase wit...
0,NOC4L,uncharacterized,7,No functional annotation was provided in the e...
1,NOL10,uncharacterized,7,No functional annotation was provided in the e...
22,PRKAR1A,novel_role,6,As a regulatory subunit of cAMP-dependent prot...
23,TRAPPC8,novel_role,6,TRAPPC8 is known for its role in ER-Golgi traf...
4,EIF3M,novel_role,6,Well-established as a component of the eIF3 tr...
6,EIF3E,novel_role,6,Well-characterized in translation initiation a...
8,EIF3D,novel_role,6,Established as the cap-binding component of eI...
2,RRP12,uncharacterized,6,No functional annotation was provided in the e...


## Test Concise CoT (Faster Alternative)

For faster analysis, use the concise CoT instructions:

In [28]:
results_concise = analyze_gene_clusters(
    input_df=cluster_df.head(2),
    model_name='claude-3-7-sonnet-20250219',
    config_dict=DEFAULT_ANTHROPIC_CONFIG,
    screen_context=ROBUST_SCREEN_CONTEXT,
    cluster_analysis_prompt=ROBUST_CLUSTER_PROMPT,
    gene_annotations_df=gene_features,
    batch_size=1,
    save_outputs=False,
    use_retrieval=True,
    knowledge_dir=knowledge_dir,
    retriever_k=10,
    cot_instructions=CONCISE_COT_INSTRUCTIONS,  # Concise version
)

print("\n=== CONCISE CoT RESULTS ===")
display(results_concise['cluster_df'])

INFO:cluster_analysis_20251019_215925.log:Processing 2 clusters with model claude-3-7-sonnet-20250219


Loaded data with 2 rows and columns: ['cluster_id', 'genes']
Created annotations dictionary with 140 entries from DataFrame


Processing clusters:   0%|          | 0/2 [00:00<?, ?it/s]

Using provided template string

INFO:cluster_analysis_20251019_215925.log:Using Anthropic Claude API


...

INFO:cluster_analysis_20251019_215925.log:Anthropic API call successful: model=claude-3-7-sonnet-20250219
INFO:cluster_analysis_20251019_215925.log:Success for cluster 21
Processing clusters:  50%|█████     | 1/2 [00:20<00:20, 20.78s/it]INFO:cluster_analysis_20251019_215925.log:Using Anthropic Claude API


Using provided template string
Appending output format instructions to template
Added 33 gene feature descriptions to prompt


INFO:cluster_analysis_20251019_215925.log:Anthropic API call successful: model=claude-3-7-sonnet-20250219
INFO:cluster_analysis_20251019_215925.log:Success for cluster 37
Processing clusters: 100%|██████████| 2/2 [00:39<00:00, 19.91s/it]
INFO:cluster_analysis_20251019_215925.log:Completed analysis for 2 clusters without saving to disk



=== CONCISE CoT RESULTS ===


,cluster_id,cluster_biological_process,pathway_confidence_level,cluster_importance_score,follow_up_suggestion,established_genes,established_gene_count,uncharacterized_genes,uncharacterized_gene_count,novel_role_genes,...,total_gene_count,highest_unchar_importance,average_unchar_importance,high_unchar_genes,high_unchar_gene_count,highest_novel_role_importance,average_novel_role_importance,high_novel_role_genes,high_novel_role_gene_count,all_cluster_genes
0,21,Ribosome biogenesis and pre-rRNA processing,High,2.64,Cluster 21 shows a strong signature of ribosom...,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;NOP1...,18,NOC4L;NOL10;RRP12,3,HK2;INO80;DYRK1A;MOB4;NEPRO;ZCCHC9;SRFBP1,...,28,7,6.67,,0,8,6.0,HK2:8,1,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;NOP1...
1,37,mTOR signaling pathway,High,2.64,Cluster 37 shows a strong enrichment for mTOR ...,MTOR;RPTOR;RICTOR;MLST8;RHEB;PDPK1;SEH1L;WDR24...,9,C7orf26,1,SLC4A7;TRAPPC8;TRAPPC11;TRAPPC3;COG2;COG3;COG4,...,17,4,4.00,,0,8,6.0,SLC4A7:8,1,MTOR;RPTOR;RICTOR;MLST8;RHEB;PDPK1;SEH1L;WDR24...


### CoT:
1. **Structured reasoning**: 6-step process for systematic analysis
2. **Evidence citation**: Explicit requirement to cite snippet numbers
3. **Verification step**: Check for contradictions and gaps
4. **Confidence calibration**: Adjust based on evidence quality
5. **Two variants**: Enhanced (thorough) and Concise (less tokens)

### Knowledge Base:
1. **Pathway-specific files**: Ribosome biogenesis, mTOR, proteasome
2. **Structured content**: Clear sections for mechanisms, genes, and novel connections